# Train the layer-9 mapper on the CV 27.0 speaker split (Colab GPU)

Trains config **(a) `linear`** (old linear stretch) or **(b) `tempo`** (option A, tempo-warped
targets), chosen by `CONFIG` below. Everything else comes from `configs/train.yaml` and is
identical for both: layer 9, batch size 4, Adam lr 1e-3, ReduceLROnPlateau (patience 5),
50 epochs, seed 42, severity `severe`.

Before the first run, upload the four files from `~/data/cv-th/colab/` on the Mac
(`cv27_embeddings_layer09.tar`, `cv27_mel_cache.tar`, `cv27_metadata.tar.gz`, `SHA256SUMS`)
to `DRIVE_DATA` in Google Drive.

The data is copied from Drive to `/content` and trained from there. A full resumable
checkpoint (`latest.pt`: model, optimizer, scheduler, epoch, best val loss, RNG state) is
written to Drive after every epoch. If the session restarts, run all cells again: training
resumes from `latest.pt` automatically.

In [ ]:
# ---- settings -------------------------------------------------------------
CONFIG = "tempo"          # "linear" = (a) old linear stretch, "tempo" = (b) option A

REPO_URL = "https://github.com/KankaveeRamsri/thai-dsr.git"
REF = "w1-speaker-split"  # branch, tag or commit sha to train

DRIVE_ROOT = "/content/drive/MyDrive/thai-dsr"
DRIVE_DATA = f"{DRIVE_ROOT}/cv27_data"                      # the uploaded archives
DRIVE_RUN = f"{DRIVE_ROOT}/runs/cv27_layer9_{CONFIG}"      # checkpoints and logs

DATA_ROOT = "/content/cv27"   # local copy of the data (fast disk, not Drive)
REPO_DIR = "/content/thai-dsr"
ALLOW_COMMIT_CHANGE = False   # refuse to resume a run with code from a different commit

assert CONFIG in ("linear", "tempo"), CONFIG

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| torch", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REF], check=True)
branch = subprocess.run(["git", "-C", REPO_DIR, "symbolic-ref", "-q", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if branch:  # a branch: bring it up to date; a commit sha stays detached as given
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--quiet", "--hard", f"origin/{branch}"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
os.chdir(REPO_DIR)
print("repo", REPO_DIR, "| ref", REF, "| commit", COMMIT)

In [ ]:
# torch is preinstalled on Colab and satisfies requirements.txt, so it is not reinstalled.
!pip install -q -r requirements.txt

In [ ]:
import hashlib
import shutil
import time

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()

ARCHIVES = ["cv27_embeddings_layer09.tar", "cv27_mel_cache.tar", "cv27_metadata.tar.gz"]
expected = {name: digest for digest, name in
            (line.split() for line in open(f"{DRIVE_DATA}/SHA256SUMS") if line.strip())}
marker = f"{DATA_ROOT}/.extracted"
if not os.path.exists(marker):
    local = "/content/archives"
    os.makedirs(local, exist_ok=True)
    os.makedirs(f"{DATA_ROOT}/meta", exist_ok=True)
    for name in ARCHIVES:
        start = time.time()
        shutil.copy(f"{DRIVE_DATA}/{name}", f"{local}/{name}")
        assert sha256(f"{local}/{name}") == expected[name], f"checksum mismatch: {name}"
        print(f"copied + verified {name} in {time.time() - start:.0f}s")
    subprocess.run(["tar", "-xf", f"{local}/cv27_embeddings_layer09.tar", "-C", DATA_ROOT], check=True)
    subprocess.run(["tar", "-xf", f"{local}/cv27_mel_cache.tar", "-C", DATA_ROOT], check=True)
    subprocess.run(["tar", "-xzf", f"{local}/cv27_metadata.tar.gz", "-C", f"{DATA_ROOT}/meta"], check=True)
    shutil.rmtree(local)
    open(marker, "w").close()

counts = {d: len(os.listdir(f"{DATA_ROOT}/{d}"))
          for d in ("embeddings", "mel_cache/clean", "mel_cache/tempo")}
print(counts)
assert counts == {"embeddings": 3612, "mel_cache/clean": 2996, "mel_cache/tempo": 3612}, counts

# The archived metadata must match the code's tracked copies (same split, same tempo segments).
for name in ["cv27_th_speaker_split.csv", "cv27_th_splits.json",
             "cv27_th_distortion_params.csv", "cv27_th_clean_16k_index.csv"]:
    assert sha256(f"{DATA_ROOT}/meta/{name}") == sha256(f"data/manifests/{name}"), name
print("metadata matches data/manifests/ at", COMMIT[:7])

In [ ]:
import json

os.environ["DATA_ROOT"] = DATA_ROOT  # paths in data/manifests/*.csv are relative to it
subprocess.run(["python", "scripts/build_cv_training_manifest.py",
                "--manifest", f"{DATA_ROOT}/training_manifest.csv",
                "--splits", f"{DATA_ROOT}/splits.json"], check=True)
assert json.load(open(f"{DATA_ROOT}/splits.json"))["splits"] == \
       json.load(open("data/manifests/cv27_th_splits.json"))["splits"]

In [ ]:
import yaml

train_cfg = yaml.safe_load(open("configs/train.yaml"))
FIXED = {
    "layer": train_cfg["data"]["layer"], "severity": train_cfg["data"]["severity"],
    "batch_size": train_cfg["optim"]["batch_size"], "learning_rate": train_cfg["optim"]["learning_rate"],
    "optimizer": train_cfg["optim"]["optimizer"], "weight_decay": train_cfg["optim"]["weight_decay"],
    "lr_scheduler": train_cfg["optim"]["lr_scheduler"], "num_epochs": train_cfg["optim"]["num_epochs"],
    "seed": train_cfg["runtime"]["seed"],
}
print(json.dumps(FIXED, indent=2))
assert FIXED == {"layer": 9, "severity": "severe", "batch_size": 4, "learning_rate": 0.001,
                 "optimizer": "adam", "weight_decay": 0.0, "lr_scheduler": "reduce_on_plateau",
                 "num_epochs": 50, "seed": 42}, "configs/train.yaml changed: (a) and (b) must share it"

os.makedirs(DRIVE_RUN, exist_ok=True)
info_path = f"{DRIVE_RUN}/run_info.json"
info = json.load(open(info_path)) if os.path.exists(info_path) else {
    "config": CONFIG, "commit": COMMIT, "fixed": FIXED, "sessions": []}
if info["commit"] != COMMIT and os.path.exists(f"{DRIVE_RUN}/latest.pt") and not ALLOW_COMMIT_CHANGE:
    raise RuntimeError(f"{DRIVE_RUN} was started at commit {info['commit']}, now {COMMIT}. "
                       "Set REF to that commit, or ALLOW_COMMIT_CHANGE = True.")
info["sessions"].append({"started": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": GPU_NAME, "commit": COMMIT})
json.dump(info, open(info_path, "w"), indent=2)

In [ ]:
import sys

cmd = [sys.executable, "-u", "-m", "src.training.train",
       "--manifest", f"{DATA_ROOT}/training_manifest.csv",
       "--splits", "data/manifests/cv27_th_splits.json", "--fixed-splits",
       "--embeddings-dir", f"{DATA_ROOT}/embeddings",
       "--mel-cache-dir", f"{DATA_ROOT}/mel_cache", "--preload", "--num-workers", "0",
       "--alignment", CONFIG, "--device", "cuda",
       "--checkpoint-dir", f"{DRIVE_RUN}/checkpoints", "--log-dir", f"{DRIVE_RUN}/logs",
       "--best-checkpoint", f"{DRIVE_RUN}/best.pt",
       "--loss-curve", f"{DRIVE_RUN}/logs/loss_curve.png",
       "--save-latest", f"{DRIVE_RUN}/latest.pt", "--auto-resume"]
if CONFIG == "tempo":
    cmd += ["--distortion-params", "data/manifests/cv27_th_distortion_params.csv"]

print("GPU:", GPU_NAME, "| config:", CONFIG, "| commit:", COMMIT[:7])
env = dict(os.environ, TQDM_DISABLE="1")  # one line per epoch instead of progress bars
with open(f"{DRIVE_RUN}/train.log", "a") as log:
    log.write(f"\n===== session {time.strftime('%Y-%m-%d %H:%M:%S')} on {GPU_NAME} at {COMMIT}\n")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    assert process.wait() == 0, f"train.py exited with {process.returncode}"

In [ ]:
import re

latest = torch.load(f"{DRIVE_RUN}/latest.pt", map_location="cpu", weights_only=True)
print(f"config {CONFIG}: epoch {latest['epoch']}/{FIXED['num_epochs']}, "
      f"best val loss {latest['best_val_loss']:.4f} at epoch {latest.get('best_epoch')}")
times = [float(m.group(1)) for m in re.finditer(r"^epoch \d+ time ([0-9.]+)s",
                                                open(f"{DRIVE_RUN}/train.log").read(), re.M)]
if times:
    print(f"time per epoch on {GPU_NAME}: mean {sum(times) / len(times):.1f}s over {len(times)} epoch(s)")